# HTB Academy Module 322 — AI Defense

This notebook is a learner's companion to the module. It records the course ideas, the security pipeline, and small safe examples. It intentionally does not call paid/external guardrail services or contain credentials.

## How to read the notation
When a formula appears, read it aloud before interpreting it. For example, `x` is read **x**, `x'` is **x-prime**, `Σ` is **sum**, and `arg min` is **the value of the argument that minimizes**. In this module, most guardrails are predicates: a function examines text and returns a Boolean decision or a repaired value.

## 1. Introduction to AI Defense

AI applications need multiple defensive layers because misuse can happen at different points. **LLM guardrails** inspect inputs and outputs at inference time; **adversarial training** exposes a model to deceptive examples during training; and **adversarial tuning** updates behavior as attack patterns evolve. No single filter is a complete security boundary.

## 2. LLM guardrails

An input guardrail checks a user request before it reaches the model. An output guardrail checks generated text before it reaches the user. The practical pipeline is: **user → input checks → LLM → output checks → user**. Strict checks improve safety but add latency and can reject legitimate requests, so thresholds and rules require iteration.

## 3. Character-based validation

A whitelist regular expression such as `^[0-9.\+\-*/()]+$` accepts only calculator characters. The anchors `^` and `$` mean the entire string must match. This is easy to reason about, but it cannot recognize intent: an alphanumeric prompt injection can pass, while a harmless expression outside the whitelist is rejected.

## 4. Traditional content validation

Content checks use exact matches, regular expressions, or string similarity. A similarity score can be written `s(a,b)` and compared with a threshold `τ` (read **tau**): reject when `s(a,b) > τ`. These rules are understandable and fast, but wording changes can bypass them and substring rules can create false positives (for example, a banned substring inside an innocent word).

## 5. AI-based guardrails

An LLM-as-a-judge receives text plus a tightly constrained instruction such as `safe` or `violation`. The decision is still a predicate, but the judge uses context rather than only characters. This can improve coverage while adding a second model call and therefore latency. A conservative implementation accepts only an exact/safe answer and rejects ambiguous judge output. Because the judge can also be manipulated, combining AI checks with deterministic checks is stronger than relying on either alone.

## 6. Guardrail libraries

Libraries such as `guardrails-ai` package recurring validators: unusual prompts, jailbreaks, profanity, secrets, and HTML sanitization. A `Guard` is a composition of validators. The `on_fail` policy is a small control-flow function: `exception` stops processing, `noop` allows the value, `fix` returns a repaired value, `reask` asks the model to regenerate, and `fix_reask` does both. The important engineering practice is to inspect validator source code before treating a library as a security boundary.

## 7. Guardrail services

An external service such as Model Armor can inspect both sides of the model interaction. The prompt is sanitized before inference; the response is checked before delivery. A service result usually has two layers: an invocation status (did the check run successfully?) and a match status (did a policy match?). Code should test both.

The service may report categories such as responsible-AI content, sensitive-data protection, prompt injection/jailbreak, malicious URI, CSAM, and virus scan. In a production pipeline, fail closed when the service cannot run, but choose a documented availability policy because fail-closed behavior can also deny service.

In [ ]:
# A local, dependency-free sketch of the two-status service contract.
# This does not contact Model Armor; it is safe to run offline.

def accept_sanitization(invocation_succeeded: bool, policy_match_found: bool) -> bool:
    """Accept only when the guardrail ran and found no policy match."""
    return invocation_succeeded and not policy_match_found

assert accept_sanitization(True, False) is True
assert accept_sanitization(True, True) is False
assert accept_sanitization(False, False) is False  # fail closed


## 10. Understanding adversarial vulnerability

FGSM reverses the usual training direction. Read the equation `x_adv = x + epsilon * sign(gradient_x L(f(x), y))` aloud as: **x-adversarial equals x plus epsilon times the sign of the gradient with respect to x of the loss of f of x and y**. `x` is the original input, `y` is the label, `L` is the loss, `gradient_x` is the direction in pixel-space that increases loss fastest, `sign` keeps only each component's direction, and `epsilon` is the maximum per-pixel step.

The infinity norm, read **the L-infinity norm**, is the largest absolute coordinate change: `||delta||_infinity = max_i |delta_i|`. If epsilon is 0.3, every pixel changes by at most 0.3. Example: a gradient `[ -2, 0.1, 5 ]` has sign `[-1, +1, +1]`; with epsilon 0.1 the perturbation is `[-0.1, +0.1, +0.1]`.

Standard training only constrains behavior near observed examples. In 784-dimensional MNIST space, the epsilon-neighborhood contains vastly more points than the training set. I-FGSM recomputes the gradient over several smaller steps, so resistance to one FGSM step does not prove a robust boundary. Training at only epsilon 0.3 can also cause epsilon overfitting; evaluate a spread such as 0.1, 0.2, 0.3, 0.4, and 0.5.

In [ ]:
# Tiny offline illustration of the FGSM formula; no model or network required.
import numpy as np

x = np.array([0.20, 0.50, 0.90])
gradient = np.array([-2.0, 0.1, 5.0])
epsilon = 0.10
delta = epsilon * np.sign(gradient)  # implements epsilon * sign(gradient_x L)
x_adv = np.clip(x + delta, 0.0, 1.0)
assert np.max(np.abs(x_adv - x)) <= epsilon
x_adv


## 11. The adversarial training defense

Adversarial training is a min-max problem. Read `min_theta E_(x,y)~D [ max_(x-prime in B-epsilon(x)) L(f_theta(x-prime),y) ]` as: **choose weights theta to minimize the average worst-case loss, where the attacker may choose any x-prime inside the epsilon neighborhood of x**. The inner `max` is the attacker; the outer `min` is the learner. FGSM approximates the inner search with one gradient-sign step.

The training loop therefore is: generate current adversarial examples, concatenate clean and adversarial inputs, duplicate the original labels, then run ordinary forward/backward optimization. A 50/50 clean/adversarial mix emphasizes robustness. Sampling epsilon from a spread avoids epsilon overfitting. Lower learning rates, weight decay, clipping, and longer training help stabilize the more difficult combined loss, at a cost of roughly 2–3x compute and memory.

In [ ]:
# Shape-only pseudocode: the comments map each line to the min-max loop.
# adv_images = fgsm(model, images, labels, epsilon)  # approximate inner max
# combined_images = torch.cat([images, adv_images], dim=0)
# combined_labels = torch.cat([labels, labels], dim=0)
# loss = cross_entropy(model(combined_images), combined_labels)
# loss.backward(); optimizer.step()  # outer minimization over theta


## 13. Training the robust model

Evaluation examples are generated once from the vulnerable baseline so every later model is judged on the same tensors. The training loop samples one epsilon per batch, creates adversarial inputs with the current weights, concatenates clean and adversarial examples, shuffles them, and minimizes cross-entropy. Saving attack tensors in CPU memory and using bounded batches avoids unnecessary GPU/RAM pressure.

## 14. Evaluation and results

The evaluator loads fixed FGSM and I-FGSM tensors and reports clean accuracy, adversarial accuracy, and attack success. Read `attack success = 100% - accuracy`: 52% I-FGSM accuracy means the attack fooled 48% of examples. A strong defense degrades gradually across epsilon and keeps the I-FGSM-versus-FGSM gap modest. A sharp drop after the training epsilon signals epsilon overfitting.

## 15. Introduction to LLM adversarial tuning

Guardrails are an application-layer filter; adversarial tuning teaches the model itself to recognize and refuse attacks. Jailbreaks try to override safety instructions through roleplay, encoding, or multi-turn context. Priming attacks inject the beginning of a harmful answer and ask the model to continue. The training set must contain both malicious prompts with safe refusals and benign prompts with helpful answers, or the model may over-refuse legitimate users.

## 16. Understanding attack vectors

Jailbreaks use roleplay, authority claims, hypothetical framing, or encoding to persuade the model to ignore safety training. Priming is different: an attacker injects a harmful-looking assistant prefix, exploiting autoregressive next-token prediction. The model treats the apparent prior assistant text as legitimate context and continues it.

A priming-defense example contains a harmful request, the apparent harmful prefix, and a stopping response. The stopping response should identify the problem and refuse. Training this pattern teaches the model to interrupt continuation, but short or novel prefixes can still evade it, so guardrails and output filtering remain necessary.

## 17. Supervised fine-tuning and LoRA

Read `L = -sum_(i=1..n) log P(x_i | x_1..x_(i-1); theta)` as: **negative sum of the log probability assigned to each correct next token**. Higher probability means lower loss, so gradient descent teaches refusal tokens to become more likely after attack-like context.

LoRA reads `W-prime = W + B A`: frozen weights plus a low-rank update. With `W` shaped `d x k`, `B` shaped `d x r`, and `A` shaped `r x k`, the adapter parameter count is `r(d+k)` instead of `dk`. For `d=k=4096` and `r=16`, that is 131,072 adapter parameters versus 16,777,216 full-weight parameters for one matrix—about 128 times fewer. Balanced jailbreak, priming, and benign examples are essential to avoid over-refusal.

In [ ]:
# Toy LoRA parameter-count calculation; no model download required.
d = k = 4096
rank = 16
full = d * k
adapter = rank * (d + k)
print({"full": full, "adapter": adapter, "reduction": full / adapter})
assert adapter < full


## 18. Lab setup and data preparation

The lab uses 4-bit quantization plus LoRA to fit a 1B-parameter Llama model on a consumer GPU. The dataset contains 96 jailbreak refusals, 42 priming defenses, and 86 benign conversations. Chat-template boundary tokens tell the tokenizer which text is the user turn and which is the assistant turn. Priming records deliberately place the harmful prefix in the apparent assistant turn before the stopping response. The roughly 60/40 safety-to-benign mix improves refusal while limiting over-refusal.

# Expanded walkthrough: Sections 10–18

## Section 10 — why ordinary models are vulnerable

FGSM is written `x_adv = x + epsilon * sign(gradient_x L(f(x), y))`. Say it aloud as: **x-adversarial equals x plus epsilon times the sign of the gradient with respect to x of the loss of f of x and y**. The gradient is a vector with one component per input pixel. A positive component says increasing that pixel would increase loss; a negative component says decreasing it would increase loss. `sign` throws away magnitude and keeps only plus or minus, so every changed coordinate receives the same-sized step.

The constraint is `||x_adv - x||_infinity <= epsilon`. Say **the infinity norm of x-adversarial minus x is at most epsilon**. The infinity norm is the largest absolute coordinate change. Example: if `delta = [0.1, -0.04, 0.07]`, then `||delta||_infinity = max(0.1, 0.04, 0.07) = 0.1`. FGSM with epsilon 0.1 stays inside that budget. A standard model has only seen a tiny sample of the 784-dimensional neighborhood around an MNIST image, so its boundary can twist unpredictably between examples. I-FGSM repeats smaller steps and re-evaluates the gradient, exposing whether the apparent robustness was only against one particular step.

## Section 11 — the adversarial-training objective

Standard training can be read as `min over theta E[L(f_theta(x), y)]`: **choose weights theta to minimize average clean loss**. Adversarial training changes it to `min_theta E[ max_(x-prime in B_epsilon(x)) L(f_theta(x-prime), y) ]`: **choose weights to minimize the average worst loss from any nearby x-prime**. The inner maximum represents an attacker searching for the worst allowed input; the outer minimum represents the model learning to make that worst case less bad.

The implementation approximates the inner maximum with FGSM, concatenates clean and adversarial tensors, duplicates labels because the image's true digit has not changed, and performs ordinary cross-entropy training. Epsilon spread training samples several budgets instead of teaching the model one memorized radius. The tradeoff is not mysterious: more adversarial examples consume capacity and compute, so clean accuracy can fall slightly while robust accuracy rises.

## Section 12 — what each component is doing

LeNet's tensor shapes are the key bookkeeping. A batch begins as `[batch, 1, 28, 28]`. `conv1` produces six feature maps while padding keeps 28 by 28; 2-by-2 pooling makes them 14 by 14. `conv2` produces sixteen maps and pooling reduces them to 5 by 5, giving `16*5*5 = 400` values per image before the linear layers. The final `[batch, 10]` tensor contains logits, not probabilities.

In `images_copy = images.clone().detach().requires_grad_(True)`, clone protects the caller's batch, detach prevents an old graph from being reused, and requires-grad asks autograd for derivatives with respect to pixels. `model.zero_grad()` prevents parameter gradients from previous work contaminating the attack. `torch.clamp` converts the normalized pixel limits from raw `[0,1]` using `(pixel - mean) / std`; for MNIST the bounds are approximately `[-0.4242, 2.8215]`. I-FGSM uses `alpha = epsilon / steps`, then projects `adv - original` back into `[-epsilon, epsilon]` after every step. Without that projection, ten legal-looking steps could add up to a perturbation larger than the stated budget.

## Section 13 — reproducible evaluation and training

The fixed `adv_examples.safetensors` file separates attack generation from model comparison: every candidate sees the same 500 clean images and the same FGSM/I-FGSM examples at each epsilon. The nested dictionaries are flattened because safetensors stores named tensors, not arbitrary Python containers. During training, a fresh epsilon is sampled per batch, so the model continuously co-evolves with attacks rather than memorizing stale examples. Moving generated tensors back to CPU and using bounded batches are memory controls, not mathematical changes.

## Section 14 — reading the evaluator

If 500 examples produce 488 correct predictions, accuracy is `488/500 * 100 = 97.6%`; attack success is `100 - 97.6 = 2.4%`. Compare the complete epsilon curve, not only epsilon 0.3. A steep cliff immediately after the training epsilon is evidence of epsilon overfitting. I-FGSM being a few percentage points worse than FGSM is expected because repeated gradients search more carefully; a large gap suggests the model learned an FGSM-specific artifact.

## Section 15 — tuning an LLM rather than filtering it

Guardrails are explicit application predicates: inspect, reject, repair. Adversarial tuning changes the model's learned conditional distribution so refusal becomes likely when context resembles an attack. The desired behavior is conditional, not blanket refusal: refuse the harmful request, but answer a legitimate question about the same broad topic. This is why the benign examples are as important as attack examples.

## Section 16 — priming

A normal jailbreak tries to influence the model before it decides how to answer. A priming attack inserts text in the assistant's apparent response position. Because an autoregressive model predicts the next token from all previous tokens, it may continue the harmful pattern as if it had generated the prefix itself. Training examples therefore include a harmful request, an apparent harmful prefix, and a stopping response that identifies the problem and refuses.

## Section 17 — SFT and LoRA math

For tokens `x_1` through `x_n`, `L = -sum log P(x_i | x_1,...,x_(i-1); theta)`. Say **loss equals negative the sum of the log probability of x sub i given all preceding tokens and parameters theta**. If the model assigns probabilities `[0.8, 0.5, 0.9]`, then `L = -(log .8 + log .5 + log .9)`, about 0.94. Improving any correct-token probability lowers loss.

LoRA's `W-prime = W + B A` means **new weight equals frozen weight plus adapter B times adapter A**. If `W` is `d` by `k`, `B` is `d` by `r`, and `A` is `r` by `k`, the adapter stores `r(d+k)` numbers instead of `dk`. With `d=k=4096` and `r=16`, that is 131,072 versus 16,777,216 parameters for one matrix. The rank restriction assumes the useful behavior change lies in a smaller subspace.

## Section 18 — data and hardware

The 224 examples are 96 jailbreak refusals, 42 priming defenses, and 86 benign conversations. The Llama chat-template markers are structural delimiters, not decoration: they tell the tokenizer where the user turn ends and the assistant turn begins. The priming formatter deliberately puts harmful-prefix text after the assistant marker, then appends the stopping response. Four-bit quantization reduces storage and activation pressure; LoRA means gradients update adapters rather than billions of base weights. The lab's CUDA requirement is genuine for the full run, so the local notebook keeps training opt-in and avoids downloading model weights automatically.

## 19. Training the safety-tuned model

Four-bit loading stores each weight in about half a byte instead of four bytes for FP32. It does not change the LoRA equation; it makes the frozen base model affordable to load. `r=16` sets the adapter rank, `lora_alpha=32` scales its update, and dropout 0.05 regularizes the adapter. Query/key/value/output projections affect how context is attended to; feed-forward projections help turn that context into refusal behavior.

The effective batch size is `per_device_batch_size * gradient_accumulation_steps = 4 * 2 = 8`. The GPU processes four examples, accumulates gradients, processes four more, then updates the adapters once. Warmup gradually increases the learning rate for the first 10% of steps; decay then reduces it toward zero. A falling negative-log-likelihood loss means the adapter is fitting the training responses, but a held-out evaluator is required to distinguish generalization from memorization.

The notebook deliberately does not download Llama or invoke `trainer.train()` automatically. If CUDA memory is tight, reduce batch size before sequence length; reducing sequence length can truncate examples. Save only adapter weights and tokenizer files, because the base model remains frozen and is reloaded at inference time.

## 20. Evaluation and validation

Validation uses held-out examples, not the training records. The three metrics measure different failure modes: jailbreak refusal asks whether manipulation prompts are declined; priming defense asks whether the model interrupts an already-started harmful response; benign helpfulness asks whether legitimate answers remain long enough, multi-sentence, and non-refusal.

A percentage is a count divided by the category size. For example, 29 defended priming cases out of 42 gives `29/42 * 100 = 69.0%`. The course's comparison—jailbreak refusal 83.3% to 100%, priming defense 7.1% to 69%, helpfulness unchanged at 100%—shows why all three metrics must be reported together. The evaluator's phrase matching is adequate for a lab regression check but can miss subtle refusals or accept shallow ones; production evaluation should add semantic review and human sampling.

## 21. Skills Assessment

The assessment combines the module's defenses and attack ideas. Three SnackOverflow chatbot variants protect an internal service token. Challenges 1 and 2 are required; challenge 3 is optional and intentionally much stronger. The token is also the flag. The assessment tests whether we can reason about a real request/response pipeline rather than rely on one filter: inspect the interface, identify which layer transforms or rejects input, and keep notes on evidence. A target address and explicit authorization are required before any live interaction.